<a href="https://colab.research.google.com/github/shivanvithajayam/1BF24CS127_AI_LAB/blob/main/resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from itertools import count

# ============================================================
# FIRST-ORDER LOGIC CNF CONVERTER + RESOLUTION PROVER
# Supports: predicates, constants, variables, function terms,
#           AND, OR, NOT, implication, biconditional,
#           forall, exists, unification and resolution.
# ============================================================

# --------------------------- TOKENIZER -----------------------

def tokenize(text):
    replacements = {
        "∀": " forall ", "∃": " exists ",
        "∧": " AND ", "∨": " OR ", "¬": " NOT ",
        "→": " -> ", "↔": " <-> ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    raw = re.findall(
        r"<->|->|[(),.&|~!:]|[A-Za-z_][A-Za-z_0-9]*",
        text
    )
    tokens = []
    for token in raw:
        upper = token.upper()
        if upper == "AND":
            tokens.append("&")
        elif upper == "OR" or token == "V":
            tokens.append("|")
        elif upper == "NOT":
            tokens.append("~")
        elif upper == "FORALL":
            tokens.append("forall")
        elif upper == "EXISTS":
            tokens.append("exists")
        else:
            tokens.append(token)
    return tokens


# ---------------------------- PARSER -------------------------

class Parser:
    def __init__(self, text):
        self.tokens = tokenize(text)
        self.pos = 0
        self.bound = set()

    def peek(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def take(self):
        token = self.peek()
        if token is None:
            raise ValueError("Unexpected end of formula")
        self.pos += 1
        return token

    def accept(self, token):
        if self.peek() == token:
            self.take()
            return True
        return False

    def parse(self):
        if not self.tokens:
            raise ValueError("Formula cannot be empty")
        result = self.parse_iff()
        if self.peek() is not None:
            raise ValueError("Unexpected token: " + self.peek())
        return result

    # Precedence: NOT > AND > OR > IMPLIES > IFF
    def parse_iff(self):
        node = self.parse_imp()
        while self.accept("<->"):
            node = ("iff", node, self.parse_imp())
        return node

    def parse_imp(self):
        node = self.parse_or()
        if self.accept("->"):
            node = ("imp", node, self.parse_imp())
        return node

    def parse_or(self):
        node = self.parse_and()
        while self.accept("|"):
            node = ("or", node, self.parse_and())
        return node

    def parse_and(self):
        node = self.parse_unary()
        while self.accept("&"):
            node = ("and", node, self.parse_unary())
        return node

    def parse_unary(self):
        if self.accept("~"):
            return ("not", self.parse_unary())

        if self.peek() in ("forall", "exists"):
            quantifier = self.take()
            variable = self.take()
            if not re.fullmatch(r"[A-Za-z_]\w*", variable):
                raise ValueError("Invalid quantified variable: " + variable)
            self.accept(".") or self.accept(":")
            old_bound = self.bound.copy()
            self.bound.add(variable)
            body = self.parse_unary()
            self.bound = old_bound
            return (quantifier, variable, body)

        if self.accept("("):
            node = self.parse_iff()
            if not self.accept(")"):
                raise ValueError("Missing closing parenthesis")
            return node

        return self.parse_atom()

    def parse_term(self):
        name = self.take()
        if not re.fullmatch(r"[A-Za-z_]\w*", name):
            raise ValueError("Invalid term: " + name)

        if self.accept("("):
            args = []
            if not self.accept(")"):
                while True:
                    args.append(self.parse_term())
                    if self.accept(")"):
                        break
                    if not self.accept(","):
                        raise ValueError("Expected ',' in function arguments")
            return ("func", name, tuple(args))

        # Conventional notation: bound names and single lowercase letters
        # are variables; names such as John, Apple, Peanuts are constants.
        if name in self.bound or re.fullmatch(r"[a-z]", name):
            return "?" + name
        return name

    def parse_atom(self):
        predicate = self.take()
        if not re.fullmatch(r"[A-Za-z_]\w*", predicate):
            raise ValueError("Expected a predicate or proposition, got: " + predicate)

        args = []
        if self.accept("("):
            if not self.accept(")"):
                while True:
                    args.append(self.parse_term())
                    if self.accept(")"):
                        break
                    if not self.accept(","):
                        raise ValueError("Expected ',' between predicate arguments")
        return ("atom", predicate, tuple(args))


def parse(text):
    return Parser(text).parse()


# ------------------------ AST HELPERS ------------------------

def is_var(term):
    return isinstance(term, str) and term.startswith("?")


def contains_op(node, operations):
    if not isinstance(node, tuple) or not node:
        return False
    op = node[0]
    if op in operations:
        return True
    if op == "not":
        return contains_op(node[1], operations)
    if op in ("and", "or", "imp", "iff"):
        return contains_op(node[1], operations) or contains_op(node[2], operations)
    if op in ("forall", "exists"):
        return contains_op(node[2], operations)
    return False


# ------------------- CNF TRANSFORMATION HELPERS --------------

def eliminate_implications(node):
    op = node[0]
    if op == "atom":
        return node
    if op == "not":
        return ("not", eliminate_implications(node[1]))
    if op == "imp":
        a = eliminate_implications(node[1])
        b = eliminate_implications(node[2])
        return ("or", ("not", a), b)
    if op == "iff":
        a = eliminate_implications(node[1])
        b = eliminate_implications(node[2])
        return ("and", ("or", ("not", a), b), ("or", ("not", b), a))
    if op in ("and", "or"):
        return (op, eliminate_implications(node[1]), eliminate_implications(node[2]))
    if op in ("forall", "exists"):
        return (op, node[1], eliminate_implications(node[2]))
    return node


def to_nnf(node, negated=False):
    op = node[0]
    if op == "atom":
        return ("not", node) if negated else node
    if op == "not":
        return to_nnf(node[1], not negated)
    if op in ("and", "or"):
        new_op = op if not negated else ("or" if op == "and" else "and")
        return (new_op, to_nnf(node[1], negated), to_nnf(node[2], negated))
    if op in ("forall", "exists"):
        new_op = op if not negated else ("exists" if op == "forall" else "forall")
        return (new_op, node[1], to_nnf(node[2], negated))
    raise ValueError("Implications must be eliminated before NNF")


standard_counter = count(1)
skolem_counter = count(1)


def replace_term(term, substitution):
    if is_var(term):
        if term in substitution:
            return replace_term(substitution[term], substitution)
        return term
    if isinstance(term, tuple) and term[0] == "func":
        return ("func", term[1], tuple(replace_term(a, substitution) for a in term[2]))
    return term


def substitute_ast(node, substitution):
    op = node[0]
    if op == "atom":
        return ("atom", node[1], tuple(replace_term(a, substitution) for a in node[2]))
    if op == "not":
        return ("not", substitute_ast(node[1], substitution))
    if op in ("and", "or"):
        return (op, substitute_ast(node[1], substitution), substitute_ast(node[2], substitution))
    if op in ("forall", "exists"):
        return (op, node[1], substitute_ast(node[2], substitution))
    return node


def rename_term_for_standardization(term, mapping):
    if is_var(term):
        return mapping.get(term, term)
    if isinstance(term, tuple) and term[0] == "func":
        return ("func", term[1], tuple(rename_term_for_standardization(a, mapping) for a in term[2]))
    return term


def standardize_apart(node, mapping=None):
    if mapping is None:
        mapping = {}
    op = node[0]
    if op == "atom":
        return ("atom", node[1], tuple(rename_term_for_standardization(a, mapping) for a in node[2]))
    if op == "not":
        return ("not", standardize_apart(node[1], mapping))
    if op in ("and", "or"):
        return (op, standardize_apart(node[1], mapping), standardize_apart(node[2], mapping))
    if op in ("forall", "exists"):
        new_name = "?v" + str(next(standard_counter))
        new_mapping = mapping.copy()
        new_mapping["?" + node[1]] = new_name
        return (op, new_name[1:], standardize_apart(node[2], new_mapping))
    return node


def skolemize(node, universal_variables=None):
    if universal_variables is None:
        universal_variables = []
    op = node[0]
    if op in ("atom", "not"):
        return node
    if op in ("and", "or"):
        return (op, skolemize(node[1], universal_variables), skolemize(node[2], universal_variables))
    if op == "forall":
        return skolemize(node[2], universal_variables + ["?" + node[1]])
    if op == "exists":
        name = "SK" + str(next(skolem_counter))
        if universal_variables:
            skolem_term = ("func", name, tuple(universal_variables))
        else:
            skolem_term = name
        body = substitute_ast(node[2], {"?" + node[1]: skolem_term})
        return skolemize(body, universal_variables)
    return node


def drop_universal_quantifiers(node):
    op = node[0]
    if op == "forall":
        return drop_universal_quantifiers(node[2])
    if op in ("and", "or"):
        return (op, drop_universal_quantifiers(node[1]), drop_universal_quantifiers(node[2]))
    return node


def distribute_and_over_or(node):
    if node[0] not in ("and", "or"):
        return node
    left = distribute_and_over_or(node[1])
    right = distribute_and_over_or(node[2])
    if node[0] == "and":
        return ("and", left, right)
    if left[0] == "and":
        return ("and",
                distribute_and_over_or(("or", left[1], right)),
                distribute_and_over_or(("or", left[2], right)))
    if right[0] == "and":
        return ("and",
                distribute_and_over_or(("or", left, right[1])),
                distribute_and_over_or(("or", left, right[2])))
    return ("or", left, right)


def needs_distribution(node):
    if not isinstance(node, tuple) or not node:
        return False
    op = node[0]
    if op == "or":
        if node[1][0] == "and" or node[2][0] == "and":
            return True
        return needs_distribution(node[1]) or needs_distribution(node[2])
    if op == "and":
        return needs_distribution(node[1]) or needs_distribution(node[2])
    return False


# ------------------------- CLAUSE EXTRACTION -----------------

def get_literals(node):
    if node[0] == "or":
        return get_literals(node[1]) + get_literals(node[2])
    if node[0] == "not" and node[1][0] == "atom":
        atom = node[1]
        return [(False, atom[1], atom[2])]
    if node[0] == "atom":
        return [(True, node[1], node[2])]
    raise ValueError("Formula is not in CNF literal form")


def get_clauses(node):
    if node[0] == "and":
        return get_clauses(node[1]) + get_clauses(node[2])
    return [get_literals(node)]


def convert_formula_set_to_cnf(formulas):
    global standard_counter, skolem_counter
    standard_counter = count(1)
    skolem_counter = count(1)

    nodes = [parse(item) if isinstance(item, str) else item for item in formulas]

    if any(contains_op(node, {"imp", "iff"}) for node in nodes):
        nodes = [eliminate_implications(node) for node in nodes]

    nodes = [to_nnf(node) for node in nodes]

    if any(contains_op(node, {"forall", "exists"}) for node in nodes):
        nodes = [standardize_apart(node) for node in nodes]

    if any(contains_op(node, {"exists"}) for node in nodes):
        nodes = [skolemize(node) for node in nodes]

    if any(contains_op(node, {"forall"}) for node in nodes):
        nodes = [drop_universal_quantifiers(node) for node in nodes]

    if any(needs_distribution(node) for node in nodes):
        nodes = [distribute_and_over_or(node) for node in nodes]

    all_clauses = []
    for node in nodes:
        all_clauses.extend(get_clauses(node))

    return all_clauses


# --------------------------- UNIFICATION ---------------------

def occurs(variable, term, substitution):
    term = replace_term(term, substitution)
    if variable == term:
        return True
    if isinstance(term, tuple) and term[0] == "func":
        return any(occurs(variable, arg, substitution) for arg in term[2])
    return False


def unify_terms(a, b, substitution):
    a = replace_term(a, substitution)
    b = replace_term(b, substitution)
    if a == b:
        return True
    if is_var(a):
        if occurs(a, b, substitution):
            return False
        substitution[a] = b
        return True
    if is_var(b):
        if occurs(b, a, substitution):
            return False
        substitution[b] = a
        return True
    if isinstance(a, tuple) and isinstance(b, tuple):
        if a[0] != "func" or b[0] != "func" or a[1] != b[1] or len(a[2]) != len(b[2]):
            return False
        return all(unify_terms(x, y, substitution) for x, y in zip(a[2], b[2]))
    return False


def unify_literals(a, b):
    if a[0] == b[0] or a[1] != b[1] or len(a[2]) != len(b[2]):
        return None
    substitution = {}
    for left, right in zip(a[2], b[2]):
        if not unify_terms(left, right, substitution):
            return None
    return substitution


def apply_substitution_to_literal(literal, substitution):
    positive, predicate, args = literal
    return (positive, predicate, tuple(replace_term(a, substitution) for a in args))


rename_counter = count(1)


def standardize_clause_apart(clause):
    mapping = {}

    def rename(term):
        if is_var(term):
            if term not in mapping:
                mapping[term] = "?r" + str(next(rename_counter))
            return mapping[term]
        if isinstance(term, tuple) and term[0] == "func":
            return ("func", term[1], tuple(rename(a) for a in term[2]))
        return term

    return [(s, p, tuple(rename(a) for a in args)) for s, p, args in clause]


def resolve_clauses(clause1, clause2):
    clause1 = standardize_clause_apart(clause1)
    clause2 = standardize_clause_apart(clause2)
    resolvents = []

    for i, literal1 in enumerate(clause1):
        for j, literal2 in enumerate(clause2):
            substitution = unify_literals(literal1, literal2)
            if substitution is None:
                continue

            remaining = clause1[:i] + clause1[i + 1:] + clause2[:j] + clause2[j + 1:]
            resolvent = [apply_substitution_to_literal(lit, substitution) for lit in remaining]

            unique = []
            for lit in resolvent:
                if lit not in unique:
                    unique.append(lit)

            tautology = any((not sign, pred, args) in unique for sign, pred, args in unique)
            if not tautology:
                resolvents.append(unique)

    return resolvents


# ---------------------- DUPLICATE CLAUSE CHECK ---------------

def canonical_clause_key(clause):
    mapping = {}

    def normalize(term):
        if is_var(term):
            if term not in mapping:
                mapping[term] = "?v" + str(len(mapping) + 1)
            return mapping[term]
        if isinstance(term, tuple) and term == "func":
            return ("func", term[1], tuple(normalize(a) for a in term[2]))
        return term

    normalized = [(s, p, tuple(normalize(a) for a in args)) for s, p, args in clause]
    return tuple(sorted(normalized, key=repr))


# -------------------------- RESOLUTION -----------------------

def resolution_proof(premises, goal, max_clauses=1500, max_steps=10000):
    global rename_counter
    rename_counter = count(1)

    premise_clauses = convert_formula_set_to_cnf(premises)
    goal_ast = parse(goal)
    negated_goal_ast = ("not", goal_ast)
    goal_clauses = convert_formula_set_to_cnf([negated_goal_ast])

    clauses = []
    seen = set()
    for clause in premise_clauses + goal_clauses:
        key = canonical_clause_key(clause)
        if key not in seen:
            seen.add(key)
            clauses.append(clause)

    if any(len(clause) == 0 for clause in clauses):
        print("PROVED")
        return True

    step = len(clauses) + 1
    changed = True

    while changed:
        changed = False
        pair_count = len(clauses)

        for i in range(pair_count):
            for j in range(i + 1, pair_count):
                for resolvent in resolve_clauses(clauses[i], clauses[j]):
                    key = canonical_clause_key(resolvent)
                    if key in seen:
                        continue

                    seen.add(key)
                    clauses.append(resolvent)
                    changed = True

                    if not resolvent:
                        print("PROVED")
                        return True

                    step += 1
                    if len(clauses) >= max_clauses or step > max_steps:
                        print("INCONCLUSIVE (Resource limit reached)")
                        return None

    print("NOT PROVED")
    return False


# ---------------------------- MAIN ---------------------------

def main():
    try:
        raw_n = input("Enter number of premises: ").strip()
        n = int(raw_n)
        if n < 0:
            raise ValueError("Number of premises cannot be negative")

        premises = []
        for i in range(n):
            premise = input(f"Enter premise {i + 1}: ").strip()
            if not premise:
                raise ValueError(f"Premise {i + 1} cannot be empty")
            premises.append(premise)

        goal = input("Enter statement to prove: ").strip()
        if not goal:
            raise ValueError("Goal cannot be empty")

        resolution_proof(premises, goal)

    except (ValueError, RecursionError) as error:
        print("ERROR:", error)
    except KeyboardInterrupt:
        print("Program cancelled by user.")


if __name__ == "__main__":
    main()


Enter number of premises: 7
Enter premise 1: ∀x (Food(x) → Likes(John,x))
Enter premise 2: Food(Apple) ∧ Food(Vegetables)
Enter premise 3: ∀x ∀y ((Eats(x,y) ∧ ¬Killed(x)) → Food(y))
Enter premise 4: Eats(Anil,Peanuts) ∧ Alive(Anil)
Enter premise 5: ∀x (Eats(Anil,x) → Eats(Harry,x))
Enter premise 6: ∀x (¬Killed(x) → Alive(x))
Enter premise 7: ∀x (Alive(x) → ¬Killed(x))
Enter statement to prove: Likes(John,Peanuts)
PROVED
